# 根据STEMNIST中数据生成压力PWL文件

In [ ]:
from pathlib import Path

import h5py
import numpy as np


DURATION = 2.0
FRAME_COUNT = 240
FRAME_DT = DURATION / FRAME_COUNT


def write_pressure_pwl(
    pressure_sequence,
    output_path,
    duration=2.0,
    edge_time=1e-9,
):
    """
    将压力序列写为LTspice PWL。

    每帧采用零阶保持：
    一帧内保持不变，在下一帧边界前1 ns切换。
    """
    pressure_sequence = np.asarray(
        pressure_sequence,
        dtype=np.float64,
    )

    if pressure_sequence.ndim != 1:
        raise ValueError("压力序列必须是一维数组")

    if len(pressure_sequence) != 240:
        raise ValueError(
            f"原始STEMNIST样本应包含240帧，"
            f"实际为{len(pressure_sequence)}"
        )

    pressure_sequence = np.clip(
        pressure_sequence,
        0,
        255,
    )

    dt = duration / len(pressure_sequence)
    edge_time = min(edge_time, dt / 1000)

    output_path = Path(output_path)

    with output_path.open(
        "w",
        encoding="ascii",
        newline="\n",
    ) as file:
        # 第0帧从t=0开始。
        file.write(
            f"0 {pressure_sequence[0]:.9g}\n"
        )

        for frame_index, pressure in enumerate(
            pressure_sequence
        ):
            frame_end = (
                frame_index + 1
            ) * dt

            # 保持当前帧，直到帧边界前1 ns。
            file.write(
                f"{frame_end - edge_time:.12g} "
                f"{pressure:.9g}\n"
            )

            # 在帧边界切换到下一帧。
            if frame_index + 1 < len(
                pressure_sequence
            ):
                next_pressure = pressure_sequence[
                    frame_index + 1
                ]

                file.write(
                    f"{frame_end:.12g} "
                    f"{next_pressure:.9g}\n"
                )

        # 明确给出2秒结束点。
        file.write(
            f"{duration:.12g} "
            f"{pressure_sequence[-1]:.9g}\n"
        )


h5_path = Path(
    r"C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST"
    r"\first_stage\STEMNIST\extracted"
    r"\STEMNIST Dataset\RawCharacters\AT_A_1.h5"
)

row = 7
column = 9

with h5py.File(h5_path, "r") as file:
    pressure = file["pressure_data"][...]

if pressure.shape != (240, 16, 16):
    raise ValueError(
        f"压力数组形状错误：{pressure.shape}"
    )

taxel_pressure = pressure[:, row, column]

output_path = Path(
    r"C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST"
    r"\docs\neurophic_system_model_var_Rload"
    r"\PCTRL_AT_A_1_r07_c09.pwl"
)

write_pressure_pwl(
    taxel_pressure,
    output_path,
)

print("PWL生成完成：", output_path)
print("压力最小值：", taxel_pressure.min())
print("压力最大值：", taxel_pressure.max())

PWL生成完成： C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\docs\neurophic_system_model\PCTRL_AT_A_1_r07_c09.pwl
压力最小值： 81
压力最大值： 180
